# Part 8 — pass@k: An Unbiased Estimator by Hand

*Evals from First Principles, Part 8.*

Code-generation and agent evals love to report "pass@k": the chance that at least one of k sampled completions passes. The obvious formula, `1 - (1 - c/n)^k`, treats your n samples as an infinite coin you can flip forever. You only drew n of them. This notebook derives the estimator that OpenAI's Codex paper actually uses, a plain combinatorial identity over subsets drawn WITHOUT replacement, checks it against brute-force enumeration, and then proves the naive formula is systematically biased by taking an exact expectation in rational arithmetic — no Monte Carlo, no sampling error, every number reproducible on a laptop with no network and no API key.

Companion script: `pass_at_k.py`

In [ ]:
from math import comb
from itertools import combinations
from fractions import Fraction

# --- The scenario ---------------------------------------------------------
# One coding task. We ask the model for n=5 completions and run each against
# a hidden unit-test suite. c=2 of the 5 pass. We label the samples so the
# reader can see exactly which ones are correct.
N = 5                        # samples generated for the task
C = 2                        # of those, how many pass the tests
K = 2                        # budget: we would ship the best of k draws
CORRECT = [1, 1, 0, 0, 0]    # sample s0,s1 pass; s2,s3,s4 fail. sum == C

labels = " ".join(f"s{i}({'ok' if CORRECT[i] else 'x '})" for i in range(N))
print(f"One task, n={N} sampled completions, c={C} pass the unit tests, k={K}.")
print(f"  samples:  {labels}")
print(f"  p_hat = c/n = {C}/{N} = {C / N:.2f}   (empirical per-sample pass rate)")

## Step 1 — The tempting plug-in (and why it's wrong)

The obvious estimator treats each of the n samples as an independent coin flip with success rate `p_hat = c/n`, so "miss all k draws" is `(1 - c/n)^k`:

```
pass@k ~= 1 - (1 - c/n)^k
```

That models sampling WITH replacement, an infinite pool of coins at rate `p_hat`. We did not draw from an infinite pool, we drew exactly n=5 samples and c=2 of them passed. Reusing the same sample twice in one "draw of k" is not how pass@k is actually used in practice, so this estimator is biased. We'll prove that in Step 5.

In [ ]:
def plugin_pass_at_k(n, c, k):
    """The tempting WRONG estimator: pretend each draw is an independent coin
    with success rate p_hat = c/n, so 'miss all k' = (1 - c/n)^k. This models
    sampling WITH replacement and is biased for the finite pool we actually
    drew."""
    return 1.0 - (1.0 - c / n) ** k


p_hat = C / N
plug = plugin_pass_at_k(N, C, K)
print(f"pass@{K} ~= 1 - (1 - c/n)^k = 1 - (1 - {p_hat:.2f})^{K}"
      f" = 1 - {(1 - p_hat) ** K:.2f} = {plug:.3f}")

## Step 2 — The unbiased estimator (sampling WITHOUT replacement)

Instead of treating the n samples as an infinite coin, count directly over the finite pool. There are `C(n, k)` equally likely size-k subsets of the n samples we could hand to a "best of k" evaluation. A subset is all-wrong only if all k picks land in the `n - c` failing samples, and there are `C(n-c, k)` ways to do that. So:

```
P(draw is all wrong) = C(n-c, k) / C(n, k)
pass@k = 1 - C(n-c, k) / C(n, k)
```

This is the estimator the Codex paper reports. When `k > n - c` there are not even enough failing samples to fill an all-wrong subset, `C(n-c, k) = 0`, and pass@k is exactly `1.0`, as it must be.

In [ ]:
def unbiased_pass_at_k(n, c, k):
    """The Codex-paper estimator. Of C(n,k) equally likely size-k subsets of
    the n samples, exactly C(n-c, k) contain none of the c correct ones (you
    must draw all k from the n-c wrong ones). So the chance of an all-wrong
    draw is C(n-c,k)/C(n,k), and pass@k is one minus that. When k > n-c,
    C(n-c,k)=0 and the estimator is exactly 1.0, as it must be."""
    return 1.0 - comb(n - c, k) / comb(n, k)


all_wrong = comb(N - C, K)
all_subsets = comb(N, K)
unb = unbiased_pass_at_k(N, C, K)
print(f"total size-k subsets      C(n, k)   = C({N},{K}) = {all_subsets}")
print(f"all-wrong size-k subsets  C(n-c, k) = C({N - C},{K}) = {all_wrong}"
      f"   (both picks from the {N - C} failing samples)")
print(f"P(draw is all wrong) = C(n-c,k)/C(n,k) = {all_wrong}/{all_subsets}"
      f" = {all_wrong / all_subsets:.3f}")
print(f"pass@{K} = 1 - C(n-c,k)/C(n,k) = 1 - {all_wrong}/{all_subsets} = {unb:.3f}")

## Step 3 — Brute-force check: enumerate every size-k subset

The unbiased formula is a closed-form count. We can check it by literally listing every size-2 subset of the 5 samples and counting how many contain at least one correct sample (s0 or s1). `C(5,2) = 10` subsets total, this is small enough to print in full.

In [ ]:
def brute_force_pass_at_k(correct_flags, k):
    """Ground truth by enumeration: list EVERY size-k subset of the samples,
    count how many contain at least one correct sample. Returns (passed,
    total). This is what the estimator is an estimator OF."""
    idx = range(len(correct_flags))
    passed = total = 0
    for subset in combinations(idx, k):
        total += 1
        if any(correct_flags[i] for i in subset):
            passed += 1
    return passed, total


print(f"size-{K} subsets of {{s0..s{N - 1}}} that contain >=1 correct sample:")
for subset in combinations(range(N), K):
    names = "{" + ",".join(f"s{i}" for i in subset) + "}"
    hit = any(CORRECT[i] for i in subset)
    print(f"  {names:11s} -> {'PASS' if hit else 'fail'}")

passed, total = brute_force_pass_at_k(CORRECT, K)
print(f"passing subsets / total = {passed}/{total} = {passed / total:.3f}")
print(f"estimator said {unb:.3f}.  Match: {abs(passed / total - unb) < 1e-12}")

## Step 4 — pass@k across every budget k

Fix n=5, c=2 and sweep the budget k from 1 to 5, comparing all three: the biased plug-in, the unbiased combinatorial estimator, and the brute-force count. pass@k should rise monotonically with k, and once k exceeds `n - c = 3` there are not enough failing samples to fill an all-wrong subset, so pass@k must hit exactly `1.000`.

In [ ]:
print("  k   plug-in(biased)   unbiased    brute-force")
for k in range(1, N + 1):
    pl = plugin_pass_at_k(N, C, k)
    ub = unbiased_pass_at_k(N, C, k)
    pk, tk = brute_force_pass_at_k(CORRECT, k)
    print(f"  {k}      {pl:.3f}          {ub:.3f}       {pk}/{tk} = {pk / tk:.3f}")

## Step 5 — Why "unbiased": exact expectation over c ~ Binomial(n, p)

We can prove which estimator is biased without any sampling, by taking an EXACT expectation. Suppose each of the n samples independently passes with some true rate p. Then the number of passing samples c is a Binomial(n, p) random variable, and the quantity pass@k is truly trying to estimate is `1 - (1-p)^k`. Average each estimator over every possible value of c, weighted by its exact binomial probability, all in `Fraction` arithmetic so there is no floating-point rounding to hide behind, and compare to the truth.

In [ ]:
def binom_pmf(c, n, p):
    """P(exactly c successes in n) as an exact Fraction."""
    return comb(n, c) * p ** c * (1 - p) ** (n - c)


def unbiased_frac(n, c, k):
    return 1 - Fraction(comb(n - c, k), comb(n, k))


def plugin_frac(n, c, k):
    return 1 - (1 - Fraction(c, n)) ** k


def expected_over_c(estimator_frac, n, k, p):
    """Exact E[estimator] with c ~ Binomial(n, p), as a Fraction."""
    return sum((binom_pmf(c, n, p) * estimator_frac(n, c, k)
                for c in range(n + 1)), Fraction(0))


p = Fraction(2, 5)                     # true per-sample rate = 0.40
true = 1 - (1 - p) ** K                # true pass@k = 1 - (1-p)^k
e_unb = expected_over_c(unbiased_frac, N, K, p)
e_plug = expected_over_c(plugin_frac, N, K, p)

print(f"true per-sample rate p = {float(p):.2f},  true pass@{K}"
      f" = 1 - (1-p)^{K} = {float(true):.3f}")
print(f"E[unbiased estimator] = {float(e_unb):.3f}"
      f"   (exact: {e_unb})  -> matches truth")
print(f"E[plug-in  estimator] = {float(e_plug):.3f}"
      f"   (exact: {e_plug})  -> low by {float(true - e_plug):.3f}")
print("The plug-in systematically UNDER-reports pass@k; the combinatorial")
print("estimator is exactly right on average. Report the unbiased one.")

## Recap

- pass@k answers: "if I sample n completions and ship the best of k of them, what's the chance at least one passes?" It only means something once n and k are fixed and reported alongside the score.
- The tempting plug-in `1 - (1-c/n)^k` samples WITH replacement from an infinite pool. It is a biased estimator of pass@k for the finite set of n samples you actually drew.
- The unbiased estimator `1 - C(n-c,k)/C(n,k)` counts size-k subsets of the finite pool WITHOUT replacement, matched exactly by brute-force enumeration.
- Taking the EXACT expectation over `c ~ Binomial(n, p)` in rational arithmetic proves it without a single random sample: the unbiased estimator averages to the true `1 - (1-p)^k`, the plug-in averages low.
- Whenever you report pass@k from sampled completions, use the combinatorial estimator, not the plug-in. The gap is not noise, it is a systematic downward bias.

Next: golden sets at scale — when a handful of examples stops being enough, and how to build a graded set you can trust as ground truth.